<a href="https://colab.research.google.com/github/sk12ms058/ai-agent-harness/blob/main/Scaler_AI_Agent_Harness_Masterclass_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the only external package we need for this demo
!pip install -U openai --quiet

# Keep the live demo output clean
import warnings
warnings.filterwarnings("ignore")

import os
import json
from pathlib import Path
from datetime import datetime
from getpass import getpass
from openai import OpenAI

print("✅ Packages installed and imports ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.2 MB/s eta 0:00:00
✅ Packages installed and imports ready!


In [ ]:
# Enter your key once at the beginning of the class.
# getpass() keeps it hidden from the audience.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

# Current cost-sensitive model with function/tool calling support.
MODEL = "gpt-5.6-luna"

print(f"✅ OpenAI client initialized — model: {MODEL}")

Enter your OpenAI API key: ··········
✅ OpenAI client initialized — model: gpt-5.6-luna


---
# 🧰 Designing Effective Harnesses for AI Agents

## The demo we will build

We will use **one tiny returns agent** throughout.

It has only two tools:

1. `get_order(order_id)` — read an order
2. `refund_order(order_id, amount)` — move money

We will first build the naive version:

```text
User → LLM → tool request → execute immediately
```

Then we will put a **harness** between model intent and real-world execution:

```text
User → LLM → tool request → HARNESS → real-world tool
                              │
                              ├─ policy
                              ├─ approvals
                              ├─ idempotency
                              ├─ event log
                              └─ step budget
```

> **The model proposes. The harness disposes.**

### Presenter path

For the cleanest live demo:

1. Run the setup cells before the session.
2. Show the naive agent once.
3. Run it **again as if the process restarted** → double refund.
4. Reset.
5. Put the same agent behind the harness → second attempt is deduplicated.
6. Try a ₹18,500 refund → human approval is required.
7. Approve it and inspect the event log.


---
## Part 1 — Our Fake E-commerce Backend

For a masterclass demo, I don't want Stripe, Shopify, a database, and four API keys distracting us.

So we will keep the **business world deterministic and local**.

The important part is that `refund_order()` has a real side effect: it appends a payment to a durable JSON ledger.

Notice one deliberate production-like weakness:

> The order service and payment service are separate systems.

`get_order()` does **not** automatically know that a payment refund already happened.

That is exactly how duplicate side effects can emerge across distributed systems.


In [ ]:
# ══════════════════════════════════════════════════════════
# FAKE E-COMMERCE BACKEND
# ══════════════════════════════════════════════════════════

DEMO_DIR = Path("/content") if Path("/content").exists() else Path(".")
PAYMENTS_FILE = DEMO_DIR / "agent_demo_payments.json"
HARNESS_FILE = DEMO_DIR / "agent_demo_harness_state.json"

ORDERS = {
    "ORD-1001": {
        "order_id": "ORD-1001",
        "item": "Noise-cancelling headphones",
        "amount": 4999,
        "currency": "INR",
        "reason": "damaged_item",
        "eligible_for_refund": True,
    },
    "ORD-2002": {
        "order_id": "ORD-2002",
        "item": "Premium laptop",
        "amount": 18500,
        "currency": "INR",
        "reason": "damaged_item",
        "eligible_for_refund": True,
    },
    "ORD-3003": {
        "order_id": "ORD-3003",
        "item": "Wireless mouse",
        "amount": 2499,
        "currency": "INR",
        "reason": "outside_return_window",
        "eligible_for_refund": False,
    },
}


def _read_json(path: Path, default):
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _write_json(path: Path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2)


def reset_demo_state():
    """Reset both the fake payment system and our harness state."""
    _write_json(PAYMENTS_FILE, [])
    if HARNESS_FILE.exists():
        HARNESS_FILE.unlink()
    print("🧹 Demo reset: payment ledger = 0, harness state cleared")


def get_order(order_id: str) -> dict:
    """Read an order. This is a read-only tool."""
    order = ORDERS.get(order_id)
    if not order:
        return {"status": "NOT_FOUND", "order_id": order_id}
    return {"status": "FOUND", **order}


def payment_api_refund(order_id: str, amount: int) -> dict:
    """
    Simulate a REAL side effect.
    This payment API is intentionally naive: every call moves money.
    """
    ledger = _read_json(PAYMENTS_FILE, [])

    receipt = {
        "refund_id": f"RF-{len(ledger) + 1:04d}",
        "order_id": order_id,
        "amount": amount,
        "currency": "INR",
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }

    ledger.append(receipt)
    _write_json(PAYMENTS_FILE, ledger)

    print(f"      💸 PAYMENT API: refunded ₹{amount:,}  →  {receipt['refund_id']}")
    return {"status": "REFUNDED", **receipt}


def show_payment_ledger():
    ledger = _read_json(PAYMENTS_FILE, [])
    print("\n💰 PAYMENT LEDGER")
    print("─" * 55)
    if not ledger:
        print("   (empty)")
    for row in ledger:
        print(
            f"   {row['refund_id']}  |  {row['order_id']}  |  "
            f"₹{row['amount']:,}  |  {row['created_at']}"
        )
    print(f"\n   Total refund operations: {len(ledger)}")
    print(f"   Total money moved: ₹{sum(x['amount'] for x in ledger):,}")
    return ledger


reset_demo_state()

print("\n📦 Available demo orders:")
for order in ORDERS.values():
    print(
        f"   {order['order_id']} | {order['item']:<28} | "
        f"₹{order['amount']:,} | eligible={order['eligible_for_refund']}"
    )

🧹 Demo reset: payment ledger = 0, harness state cleared

📦 Available demo orders:
   ORD-1001 | Noise-cancelling headphones  | ₹4,999 | eligible=True
   ORD-2002 | Premium laptop               | ₹18,500 | eligible=True
   ORD-3003 | Wireless mouse               | ₹2,499 | eligible=False


### The dangerous line

Our payment function is intentionally simple:

```python
payment_api_refund(order_id, amount)
```

Every time it is called, money moves.

There is no intelligence in the payment API and no awareness of what the LLM intended.

That separation is important:

> **The model emits intent. Application code creates the side effect.**


---
## Part 2 — Give the LLM Tools

This should look familiar from the previous workshop.

We describe Python functions using JSON Schema.

The model can now **request** a tool call.

It still cannot execute the Python function itself.


In [ ]:
# ══════════════════════════════════════════════════════════
# TOOL DEFINITIONS — what the MODEL is allowed to request
# ══════════════════════════════════════════════════════════

agent_tools = [
    {
        "type": "function",
        "name": "get_order",
        "description": (
            "Look up an order before deciding whether to refund it. "
            "Returns the item, exact amount, and refund eligibility."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Order ID such as ORD-1001",
                }
            },
            "required": ["order_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "refund_order",
        "description": (
            "Request a full refund for an eligible order. "
            "Use the exact amount returned by get_order."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "integer"},
            },
            "required": ["order_id", "amount"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

print("✅ Tool schemas ready")
for tool in agent_tools:
    print(f"   🔧 {tool['name']}: {tool['description']}")

✅ Tool schemas ready
   🔧 get_order: Look up an order before deciding whether to refund it. Returns the item, exact amount, and refund eligibility.
   🔧 refund_order: Request a full refund for an eligible order. Use the exact amount returned by get_order.


---
## Part 3 — The Naive Agent Loop

Here is the architecture most demos start with:

```text
while not done:
    response = model(context)

    if response asks for a tool:
        execute(tool)         # ← the dangerous word
        add result to context
```

Look carefully at the executor below.

Its policy is effectively:

```python
if model_requested_tool:
    YES
```


In [ ]:
# ══════════════════════════════════════════════════════════
# NAIVE EXECUTOR — "if the model asked, execute it"
# ══════════════════════════════════════════════════════════

def naive_execute(tool_name: str, args: dict) -> dict:
    if tool_name == "get_order":
        return get_order(**args)

    if tool_name == "refund_order":
        # ⚠️ No policy check.
        # ⚠️ No approval.
        # ⚠️ No idempotency.
        # ⚠️ No durable record of "this logical action already happened".
        return payment_api_refund(**args)

    raise ValueError(f"Unknown tool requested: {tool_name}")


RETURNS_AGENT_PROMPT = """
You are an e-commerce returns agent.

WORKFLOW:
1. When the user asks for a refund and gives an order ID, call get_order first.
2. If eligible_for_refund is true, request refund_order for the EXACT amount
   returned by get_order.
3. If a tool reports that the action was blocked or needs human approval,
   explain that to the user and stop. Do not repeatedly retry a blocked tool.
4. If a tool says the action was already completed, tell the user no new
   refund was issued.
5. Never invent an order amount.

Keep the final customer-facing answer short.
""".strip()


def run_agent(user_query: str, executor, max_steps: int = 8, verbose: bool = True):
    """
    Generic agent loop.

    IMPORTANT:
    The same LLM, prompt, and tool schemas will be used for BOTH demos.
    The only thing we will swap is the executor:
        naive_execute  ->  harness.execute
    """
    conversation = [{"role": "user", "content": user_query}]

    if verbose:
        print("=" * 68)
        print("🤖 AGENT STARTED")
        print(f"👤 User: {user_query}")
        print("=" * 68)

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n🔄 Step {step}/{max_steps}")

        response = client.responses.create(
            model=MODEL,
            instructions=RETURNS_AGENT_PROMPT,
            input=conversation,
            tools=agent_tools,
            tool_choice="auto",
        )

        # Preserve the model's output items so the next turn can see
        # its own tool requests / reasoning context.
        conversation += list(response.output)

        tool_calls = [
            item for item in response.output
            if getattr(item, "type", None) == "function_call"
        ]

        # No tool request = model is done.
        if not tool_calls:
            if verbose:
                print(f"\n✅ Agent: {response.output_text}")
            return {
                "answer": response.output_text,
                "steps": step,
                "conversation": conversation,
                "stopped_by_budget": False,
            }

        for call in tool_calls:
            args = json.loads(call.arguments)

            if verbose:
                print(f"   🧠 MODEL PROPOSES: {call.name}({json.dumps(args)})")

            # THIS is the control boundary.
            result = executor(call.name, args)

            if verbose:
                print(f"   📥 TOOL RESULT: {json.dumps(result)}")

            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    print(f"\n🛑 Step budget exhausted after {max_steps} steps.")
    return {
        "answer": "Stopped by step budget.",
        "steps": max_steps,
        "conversation": conversation,
        "stopped_by_budget": True,
    }


print("✅ Naive agent loop ready")

✅ Naive agent loop ready


---
# 💥 Demo 1 — The Double Refund

Use the small order first:

> **ORD-1001 — ₹4,999 — damaged — eligible**

Run the agent once.

Then imagine the worker crashed and was restarted without durable execution state.

Run the **same request again**.

The order service still says the order is eligible. The payment system remembers the money movement, but the agent runtime does not.


In [ ]:
# First run
reset_demo_state()

request = "My headphones arrived damaged. Order ORD-1001. Please refund me."

first_run = run_agent(
    request,
    executor=naive_execute,
    verbose=True,
)

show_payment_ledger()

🧹 Demo reset: payment ledger = 0, harness state cleared
🤖 AGENT STARTED
👤 User: My headphones arrived damaged. Order ORD-1001. Please refund me.

🔄 Step 1/8
   🧠 MODEL PROPOSES: get_order({"order_id": "ORD-1001"})
   📥 TOOL RESULT: {"status": "FOUND", "order_id": "ORD-1001", "item": "Noise-cancelling headphones", "amount": 4999, "currency": "INR", "reason": "damaged_item", "eligible_for_refund": true}

🔄 Step 2/8
   🧠 MODEL PROPOSES: refund_order({"order_id": "ORD-1001", "amount": 4999})
      💸 PAYMENT API: refunded ₹4,999  →  RF-0001
   📥 TOOL RESULT: {"status": "REFUNDED", "refund_id": "RF-0001", "order_id": "ORD-1001", "amount": 4999, "currency": "INR", "created_at": "2026-08-29T14:24:33"}

🔄 Step 3/8

✅ Agent: Your refund of ₹4,999 for order ORD-1001 has been issued. Refund ID: RF-0001.

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-1001  |  ₹4,999  |  2026-08-29T14:24:33

   Total refund operations: 1
   Total money moved: ₹4,999


[{'refund_id': 'RF-0001',
  'order_id': 'ORD-1001',
  'amount': 4999,
  'currency': 'INR',
  'created_at': '2026-08-29T14:24:33'}]

In [ ]:
# 💥 "PROCESS RESTART"
#
# A fresh agent invocation has no idea the previous logical action happened.
# The order service still returns eligible_for_refund=True.

print("💥 Simulating worker restart...\n")

second_run = run_agent(
    request,
    executor=naive_execute,
    verbose=True,
)

show_payment_ledger()

💥 Simulating worker restart...

🤖 AGENT STARTED
👤 User: My headphones arrived damaged. Order ORD-1001. Please refund me.

🔄 Step 1/8
   🧠 MODEL PROPOSES: get_order({"order_id": "ORD-1001"})
   📥 TOOL RESULT: {"status": "FOUND", "order_id": "ORD-1001", "item": "Noise-cancelling headphones", "amount": 4999, "currency": "INR", "reason": "damaged_item", "eligible_for_refund": true}

🔄 Step 2/8
   🧠 MODEL PROPOSES: refund_order({"order_id": "ORD-1001", "amount": 4999})
      💸 PAYMENT API: refunded ₹4,999  →  RF-0002
   📥 TOOL RESULT: {"status": "REFUNDED", "refund_id": "RF-0002", "order_id": "ORD-1001", "amount": 4999, "currency": "INR", "created_at": "2026-08-29T14:24:48"}

🔄 Step 3/8

✅ Agent: Your refund of ₹4,999 for order ORD-1001 has been issued. Refund ID: RF-0002.

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-1001  |  ₹4,999  |  2026-08-29T14:24:33
   RF-0002  |  ORD-1001  |  ₹4,999  |  2026-08-29T14:24:48

   Total refund operations: 

[{'refund_id': 'RF-0001',
  'order_id': 'ORD-1001',
  'amount': 4999,
  'currency': 'INR',
  'created_at': '2026-08-29T14:24:33'},
 {'refund_id': 'RF-0002',
  'order_id': 'ORD-1001',
  'amount': 4999,
  'currency': 'INR',
  'created_at': '2026-08-29T14:24:48'}]

### What failed?

The LLM did exactly what we told it to do:

1. Look up the order.
2. See that it is eligible.
3. Request the refund.

The payment API also did exactly what it was asked to do.

The missing component was **outside both of them**.

```text
First run:
LLM → refund_order → ₹4,999 moved

Process restarts

Second run:
LLM → refund_order → another ₹4,999 moved
```

> The real world retained the side effect.  
> The agent runtime did not retain the identity of the logical operation.

Now we add the harness.


---
# 🧰 Part 4 — Build the Harness

Our harness will own five deterministic responsibilities:

```text
1. TOOL REGISTRY     Is this even a known tool?
2. POLICY            Is the requested action allowed?
3. HITL              Does a human need to approve it?
4. IDEMPOTENCY       Did this logical side effect already happen?
5. EVENT LOG         What exactly happened at each boundary?
```

We will also persist harness state to a JSON file, so creating a new Python `AgentHarness` object simulates a process restart.

This is the key architectural change:

```text
BEFORE
LLM → execute()

AFTER
LLM → proposed action → HARNESS → execute()
```


In [ ]:
# ══════════════════════════════════════════════════════════
# THE HARNESS
# ══════════════════════════════════════════════════════════

class AgentHarness:
    HIGH_VALUE_REFUND_LIMIT = 10_000

    def __init__(self, state_file: Path = HARNESS_FILE):
        self.state_file = Path(state_file)

        self.state = _read_json(
            self.state_file,
            {
                "idempotency": {},
                "pending_approvals": {},
                "events": [],
            },
        )

    # ──────────────────────────────────────────────────────
    # DURABLE EVENT STREAM
    # ──────────────────────────────────────────────────────
    def _save(self):
        _write_json(self.state_file, self.state)

    def _event(self, event_type: str, **data):
        event = {
            "n": len(self.state["events"]) + 1,
            "time": datetime.now().isoformat(timespec="seconds"),
            "type": event_type,
            **data,
        }
        self.state["events"].append(event)
        self._save()
        return event

    # ──────────────────────────────────────────────────────
    # CONTROL BOUNDARY
    # ──────────────────────────────────────────────────────
    def execute(self, tool_name: str, args: dict) -> dict:
        # evaluate these args..
        self._event("TOOL_REQUESTED", tool=tool_name, args=args)

        # 1) Tool registry / allowlist
        if tool_name not in {"get_order", "refund_order"}:
            self._event("TOOL_DENIED", tool=tool_name, reason="UNKNOWN_TOOL")
            return {
                "status": "BLOCKED",
                "reason": "UNKNOWN_TOOL",
            }

        # Read-only tools can execute immediately.
        if tool_name == "get_order":
            result = get_order(**args)
            self._event("TOOL_COMPLETED", tool=tool_name, result=result)
            return result

        # Everything below protects refund_order.
        order_id = args.get("order_id")
        amount = args.get("amount")
        order = ORDERS.get(order_id)

        # 2) Deterministic business-policy validation
        if not order:
            return self._deny(tool_name, args, "ORDER_NOT_FOUND")

        if not order["eligible_for_refund"]:
            return self._deny(tool_name, args, "ORDER_NOT_ELIGIBLE")

        if amount != order["amount"]:
            return self._deny(
                tool_name,
                args,
                f"AMOUNT_MISMATCH: expected {order['amount']}",
            )

        # 3) Idempotency BEFORE executing the side effect
        operation_key = f"full_refund:{order_id}"

        if operation_key in self.state["idempotency"]:
            original = self.state["idempotency"][operation_key]

            self._event(
                "TOOL_DEDUPLICATED",
                tool=tool_name,
                operation_key=operation_key,
                original_refund_id=original["refund_id"],
            )

            return {
                "status": "ALREADY_COMPLETED",
                "message": "No new refund was issued.",
                "original_result": original,
            }

        # 4) Human approval for high-consequence actions
        if amount > self.HIGH_VALUE_REFUND_LIMIT:
            approval_id = f"APR-{order_id}"
            approval = self.state["pending_approvals"].get(approval_id)

            if not approval or approval.get("status") != "APPROVED":
                self.state["pending_approvals"][approval_id] = {
                    "approval_id": approval_id,
                    "status": "PENDING",
                    "tool": tool_name,
                    "args": args,
                    "reason": (
                        f"Refund ₹{amount:,} exceeds "
                        f"₹{self.HIGH_VALUE_REFUND_LIMIT:,} limit"
                    ),
                }
                self._event(
                    "HUMAN_APPROVAL_REQUIRED",
                    approval_id=approval_id,
                    tool=tool_name,
                    args=args,
                )

                return {
                    "status": "BLOCKED",
                    "reason": "HUMAN_APPROVAL_REQUIRED",
                    "approval_id": approval_id,
                    "message": (
                        f"Refunds above ₹{self.HIGH_VALUE_REFUND_LIMIT:,} "
                        "require human approval."
                    ),
                }

        # 5) The harness, not the model, finally authorizes execution.
        self._event("TOOL_APPROVED", tool=tool_name, args=args)

        result = payment_api_refund(order_id=order_id, amount=amount)

        # Record the logical side effect durably.
        self.state["idempotency"][operation_key] = result

        # Mark approval consumed, if one existed.
        approval_id = f"APR-{order_id}"
        if approval_id in self.state["pending_approvals"]:
            self.state["pending_approvals"][approval_id]["status"] = "COMPLETED"

        self._event(
            "TOOL_COMPLETED",
            tool=tool_name,
            operation_key=operation_key,
            result=result,
        )

        return result

    def _deny(self, tool_name: str, args: dict, reason: str) -> dict:
        self._event("TOOL_DENIED", tool=tool_name, args=args, reason=reason)
        return {
            "status": "BLOCKED",
            "reason": reason,
        }

    # ──────────────────────────────────────────────────────
    # DURABLE HUMAN-IN-THE-LOOP
    # ──────────────────────────────────────────────────────
    def approve(self, approval_id: str) -> dict:
        approval = self.state["pending_approvals"].get(approval_id)

        if not approval:
            return {
                "status": "NOT_FOUND",
                "approval_id": approval_id,
            }

        if approval["status"] == "COMPLETED":
            return {
                "status": "ALREADY_COMPLETED",
                "approval_id": approval_id,
            }

        approval["status"] = "APPROVED"
        self._event("HUMAN_APPROVED", approval_id=approval_id)

        # Resume exactly the suspended action.
        return self.execute(
            approval["tool"],
            approval["args"],
        )

    def show_events(self, last_n: int = None):
        events = self.state["events"]
        if last_n:
            events = events[-last_n:]

        print("\n📜 HARNESS EVENT LOG")
        print("═" * 76)

        for event in events:
            detail = ""

            if "tool" in event:
                detail += f" tool={event['tool']}"

            if "approval_id" in event:
                detail += f" approval={event['approval_id']}"

            if "operation_key" in event:
                detail += f" key={event['operation_key']}"

            if "reason" in event:
                detail += f" reason={event['reason']}"

            print(
                f"{event['n']:>2}. "
                f"{event['type']:<28}"
                f"{detail}"
            )

    def show_pending_approvals(self):
        print("\n🙋 HUMAN APPROVALS")
        print("═" * 76)

        if not self.state["pending_approvals"]:
            print("   (none)")
            return

        for approval in self.state["pending_approvals"].values():
            print(
                f"   {approval['approval_id']} | "
                f"{approval['status']:<9} | "
                f"{approval['tool']}({approval['args']})"
            )


print("✅ Harness defined")
print("   • deterministic policy")
print("   • durable event log")
print("   • human approval")
print("   • idempotency / deduplication")

✅ Harness defined
   • deterministic policy
   • durable event log
   • human approval
   • idempotency / deduplication


---
# ✅ Demo 2 — Same Agent, Same Tools, Harnessed Execution

Important: **we are not improving the prompt**.

We are not switching to a stronger model.

We are not asking the model to “please remember not to double refund.”

We keep:

- the same LLM
- the same system prompt
- the same tool definitions
- the same user request

We change only:

```python
executor = naive_execute
```

to:

```python
executor = harness.execute
```


In [ ]:
# Start clean
reset_demo_state()
harness = AgentHarness()

request = "My headphones arrived damaged. Order ORD-1001. Please refund me."

first_harnessed_run = run_agent(
    request,
    executor=harness.execute,
    verbose=True,
)

show_payment_ledger()
harness.show_events()

🧹 Demo reset: payment ledger = 0, harness state cleared
🤖 AGENT STARTED
👤 User: My headphones arrived damaged. Order ORD-1001. Please refund me.

🔄 Step 1/8
   🧠 MODEL PROPOSES: get_order({"order_id": "ORD-1001"})
   📥 TOOL RESULT: {"status": "FOUND", "order_id": "ORD-1001", "item": "Noise-cancelling headphones", "amount": 4999, "currency": "INR", "reason": "damaged_item", "eligible_for_refund": true}

🔄 Step 2/8
   🧠 MODEL PROPOSES: refund_order({"order_id": "ORD-1001", "amount": 4999})
      💸 PAYMENT API: refunded ₹4,999  →  RF-0001
   📥 TOOL RESULT: {"status": "REFUNDED", "refund_id": "RF-0001", "order_id": "ORD-1001", "amount": 4999, "currency": "INR", "created_at": "2026-08-29T14:27:32"}

🔄 Step 3/8

✅ Agent: Your refund of ₹4,999 for order ORD-1001 has been issued successfully. Refund ID: RF-0001.

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-1001  |  ₹4,999  |  2026-08-29T14:27:32

   Total refund operations: 1
   Total money moved

### Now simulate the process restarting

We create a **brand-new harness object**.

Its in-memory Python object is gone.

But it reloads the durable execution state from disk.

Then we send the same user request again.

Watch for:

```text
TOOL_DEDUPLICATED
```

The LLM is still allowed to *propose* the duplicate refund.

The harness simply refuses to create the duplicate effect.


In [ ]:
# 🔄 Simulate a restarted worker / application process
print("🔄 Creating a brand-new AgentHarness from durable state...\n")

restarted_harness = AgentHarness(HARNESS_FILE)

second_harnessed_run = run_agent(
    request,
    executor=restarted_harness.execute,
    verbose=True,
)

show_payment_ledger()
restarted_harness.show_events(last_n=8)

🔄 Creating a brand-new AgentHarness from durable state...

🤖 AGENT STARTED
👤 User: My headphones arrived damaged. Order ORD-1001. Please refund me.

🔄 Step 1/8
   🧠 MODEL PROPOSES: get_order({"order_id": "ORD-1001"})
   📥 TOOL RESULT: {"status": "FOUND", "order_id": "ORD-1001", "item": "Noise-cancelling headphones", "amount": 4999, "currency": "INR", "reason": "damaged_item", "eligible_for_refund": true}

🔄 Step 2/8
   🧠 MODEL PROPOSES: refund_order({"order_id": "ORD-1001", "amount": 4999})
   📥 TOOL RESULT: {"status": "ALREADY_COMPLETED", "message": "No new refund was issued.", "original_result": {"status": "REFUNDED", "refund_id": "RF-0001", "order_id": "ORD-1001", "amount": 4999, "currency": "INR", "created_at": "2026-08-29T14:27:32"}}

🔄 Step 3/8

✅ Agent: A refund of ₹4,999 for order ORD-1001 was already issued. No new refund was created.

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-1001  |  ₹4,999  |  2026-08-29T14:27:32

   Total r

### The difference

```text
NAIVE EXECUTION

model proposes refund
        ↓
execute immediately
        ↓
money moves


HARNESS EXECUTION

model proposes refund
        ↓
validate policy
        ↓
check approval
        ↓
check operation identity
        ↓
authorize / deny / deduplicate
        ↓
money MAY move
```

> **A tool call is a proposal, not authority.**


---
# 🧑‍⚖️ Demo 3 — Human-in-the-Loop for High-Consequence Actions

Now switch orders.

**ORD-2002** is an eligible refund for **₹18,500**.

Our model's job is still simple:

> If an eligible damaged order should be refunded, request `refund_order`.

But our *organization* has a deterministic rule:

```text
Refund ≤ ₹10,000  → may execute automatically
Refund > ₹10,000  → must wait for a human
```

Notice where that rule lives:

**not in the prompt — in the harness.**


In [ ]:
# Reset so this demo has a clean trace
reset_demo_state()
harness = AgentHarness()

expensive_request = (
    "The laptop I received is damaged. "
    "Order ORD-2002. Please refund it."
)

blocked_run = run_agent(
    expensive_request,
    executor=harness.execute,
    verbose=True,
)

# Money should NOT have moved.
show_payment_ledger()

# But the intent should be durably waiting for a human.
harness.show_pending_approvals()
harness.show_events()

🧹 Demo reset: payment ledger = 0, harness state cleared
🤖 AGENT STARTED
👤 User: The laptop I received is damaged. Order ORD-2002. Please refund it.

🔄 Step 1/8
   🧠 MODEL PROPOSES: get_order({"order_id": "ORD-2002"})
   📥 TOOL RESULT: {"status": "FOUND", "order_id": "ORD-2002", "item": "Premium laptop", "amount": 18500, "currency": "INR", "reason": "damaged_item", "eligible_for_refund": true}

🔄 Step 2/8
   🧠 MODEL PROPOSES: refund_order({"order_id": "ORD-2002", "amount": 18500})
   📥 TOOL RESULT: {"status": "BLOCKED", "reason": "HUMAN_APPROVAL_REQUIRED", "approval_id": "APR-ORD-2002", "message": "Refunds above \u20b910,000 require human approval."}

🔄 Step 3/8

✅ Agent: Your refund request for ORD-2002 requires human approval because the amount is ₹18,500. No refund has been issued yet.

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   (empty)

   Total refund operations: 0
   Total money moved: ₹0

🙋 HUMAN APPROVALS
═════════════════════════════════════════

### Suspend now. Resume later.

In a real system, the human might approve this:

- 30 seconds later
- 4 hours later
- tomorrow morning

The worker that originally asked for approval might no longer exist.

So human-in-the-loop is not:

```python
input("Approve? y/n")
```

It is **durable workflow state**.

Now approve the pending action.


In [ ]:
# The approval ID is deterministic for this demo.
approval_result = harness.approve("APR-ORD-2002")

print("\n✅ HUMAN APPROVAL RESULT")
print(json.dumps(approval_result, indent=2))

show_payment_ledger()
harness.show_pending_approvals()
harness.show_events(last_n=8)

      💸 PAYMENT API: refunded ₹18,500  →  RF-0001

✅ HUMAN APPROVAL RESULT
{
  "status": "REFUNDED",
  "refund_id": "RF-0001",
  "order_id": "ORD-2002",
  "amount": 18500,
  "currency": "INR",
  "created_at": "2026-08-29T14:46:35"
}

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-2002  |  ₹18,500  |  2026-08-29T14:46:35

   Total refund operations: 1
   Total money moved: ₹18,500

🙋 HUMAN APPROVALS
════════════════════════════════════════════════════════════════════════════
   APR-ORD-2002 | COMPLETED | refund_order({'order_id': 'ORD-2002', 'amount': 18500})

📜 HARNESS EVENT LOG
════════════════════════════════════════════════════════════════════════════
 1. TOOL_REQUESTED               tool=get_order
 2. TOOL_COMPLETED               tool=get_order
 3. TOOL_REQUESTED               tool=refund_order
 4. HUMAN_APPROVAL_REQUIRED      tool=refund_order approval=APR-ORD-2002
 5. HUMAN_APPROVED               approval=APR-ORD-2002
 6. TOOL_REQUESTE

### Retry the approved refund

Maybe an HTTP response was lost.

Maybe a queue redelivered a message.

Maybe a worker restarted and replayed the operation.

The harness should make that boring.

We will directly replay the exact same tool request.

The payment ledger should remain at **one** refund.


In [ ]:
# Simulate a retry / replay of the exact same logical side effect.
retry_result = harness.execute(
    "refund_order",
    {
        "order_id": "ORD-2002",
        "amount": 18500,
    },
)

print("\n🔁 RETRY RESULT")
print(json.dumps(retry_result, indent=2))

show_payment_ledger()
harness.show_events(last_n=6)


🔁 RETRY RESULT
{
  "status": "ALREADY_COMPLETED",
  "message": "No new refund was issued.",
  "original_result": {
    "status": "REFUNDED",
    "refund_id": "RF-0001",
    "order_id": "ORD-2002",
    "amount": 18500,
    "currency": "INR",
    "created_at": "2026-08-29T11:28:11"
  }
}

💰 PAYMENT LEDGER
───────────────────────────────────────────────────────
   RF-0001  |  ORD-2002  |  ₹18,500  |  2026-08-29T11:28:11

   Total refund operations: 1
   Total money moved: ₹18,500

📜 HARNESS EVENT LOG
════════════════════════════════════════════════════════════════════════════
 9. HUMAN_APPROVED               approval=APR-ORD-2002
10. TOOL_REQUESTED               tool=refund_order
11. TOOL_APPROVED                tool=refund_order
12. TOOL_COMPLETED               tool=refund_order key=full_refund:ORD-2002
13. TOOL_REQUESTED               tool=refund_order
14. TOOL_DEDUPLICATED            tool=refund_order key=full_refund:ORD-2002


---
# 🛡️ Demo 4 — Deterministic Policy Beats Prompt Policy

Let's try an ineligible order.

**ORD-3003** is outside the return window.

Even if the model requests a refund, the harness validates application truth before money moves.

For a live class, you can either run the full agent or call the harness directly to make the trust boundary painfully obvious.


In [ ]:
reset_demo_state()
harness = AgentHarness()

# We intentionally call the harness directly.
# Think of this as: "Assume the model made the worst possible proposal."

bad_proposal = {
    "order_id": "ORD-3003",
    "amount": 2499,
}

print("🧠 MODEL PROPOSES: refund_order", bad_proposal)

result = harness.execute("refund_order", bad_proposal)

print("\n🛡️ HARNESS RETURNS:")
print(json.dumps(result, indent=2))

show_payment_ledger()
harness.show_events()

---
# 🔍 Inspect the Harness as a System

At this point we have separated four things that often get blurred together:

| Concept | In our demo |
|---|---|
| **Model intent** | `function_call` generated by the LLM |
| **Application state** | order data |
| **Real-world side effect** | payment ledger |
| **Harness state** | approvals, idempotency keys, event stream |

This is why the architecture is more useful than simply saying:

> “Give the agent memory.”

Different state has different ownership, durability, and security requirements.


In [ ]:
# Peek at the durable harness state itself.
# This is deliberately boring JSON — that is the point.

print("🧠 DURABLE HARNESS STATE\n")
print(json.dumps(harness.state, indent=2))

---
# 🎯 What Did We Actually Add?

Our original agent loop was tiny:

```python
response = model(context)
execute(response.tool_call)
```

The **intelligence** did not change.

The surrounding runtime did.

```text
                         ┌─────────────┐
                         │     LLM     │
                         └──────┬──────┘
                                │ proposes
                                ▼
                  ┌──────────────────────────┐
                  │         HARNESS          │
                  │                          │
                  │  Tool registry           │
                  │  Argument validation     │
                  │  Business policy         │
                  │  Human approval          │
                  │  Idempotency             │
                  │  Durable event log       │
                  │  Step budget             │
                  └────────────┬─────────────┘
                               │ authorizes
                               ▼
                         REAL-WORLD APIs
```

### Lines worth landing in the room

> **The model proposes. The harness disposes.**

> **Prompt engineering says what the model should do. Harness engineering controls what the system can do.**

> **If your safety guarantee depends on the model never making a bad decision, it is not a safety guarantee.**

> **Durability prevents lost progress. Idempotency prevents duplicated effects.**

> **Autonomy should decrease as consequence increases.**


---
# 🚨 Emergency Backup — If the LLM API Misbehaves During the Live Class

The harness is application code, so you can demonstrate the core lesson even if Wi-Fi/API access is flaky.

The cell below **simulates model-generated tool intents**.

You can tell the class:

> “Pretend these JSON objects are exactly what came back from the model. Remember: the model only proposes the call.”

This backup still demonstrates:

- naive duplicate side effects
- harness deduplication
- policy enforcement
- human approval


In [ ]:
# ══════════════════════════════════════════════════════════
# OFFLINE / API-FAILURE BACKUP DEMO
# ══════════════════════════════════════════════════════════

print("1️⃣ NAIVE EXECUTOR — same logical refund twice")
reset_demo_state()

proposal = {
    "order_id": "ORD-1001",
    "amount": 4999,
}

print("\n🧠 Proposed:", proposal)
naive_execute("refund_order", proposal)

print("\n💥 Process restarts; same proposal arrives again...")
naive_execute("refund_order", proposal)

show_payment_ledger()


print("\n\n2️⃣ HARNESS — same proposal twice")
reset_demo_state()
backup_harness = AgentHarness()

backup_harness.execute("refund_order", proposal)

print("\n🔄 Replaying same proposal...")
backup_harness.execute("refund_order", proposal)

show_payment_ledger()
backup_harness.show_events()


print("\n\n3️⃣ HIGH-VALUE REFUND — approval required")
expensive = {
    "order_id": "ORD-2002",
    "amount": 18500,
}

blocked = backup_harness.execute("refund_order", expensive)
print(json.dumps(blocked, indent=2))

show_payment_ledger()
backup_harness.show_pending_approvals()

---
# 🗣️ Presenter Cheat Sheet

## Cold open
Run **Demo 1** twice.

After the second payment:

> “Who made the mistake — the LLM, the payment API, our prompt, or our architecture?”

Let the room answer.

Then:

> “The model made a reasonable decision twice. The payment API correctly executed twice. Our system had no durable concept of *this logical refund already happened*.”

---

## Reveal the harness

Before running Demo 2:

> “I am not changing the model. I am not changing the prompt. I am not changing the tools. I am changing one line: who owns `execute()`.”

When `TOOL_DEDUPLICATED` appears:

> “The model is free to be wrong. The system is not free to move money twice.”

---

## Human approval demo

When the ₹18,500 request gets blocked:

> “Notice: the LLM did not decide that this refund needed human approval. My organization's deterministic policy did.”

Then approve it:

> “The approval is state. That means it can survive the worker that requested it.”

---

## Senior-engineer callback

After the retry:

> “I am casually calling this exactly-once behaviour in the demo, but the precise production claim is safer retries through idempotency/deduplication. Exactly-once effects across distributed systems need much more careful semantics.”

---

## Transition back to the deck

> “And this is the harness pattern. Today we implemented policy, HITL, idempotency, durability, events and budgets in a few lines. Production frameworks package these capabilities differently, but the architecture is the same: probabilistic intent inside deterministic control boundaries.”
